# 1.转换字符串

In [1]:
import os

from langchain_core.output_parsers import StrOutputParser, JsonOutputParser
from langchain_openai import ChatOpenAI

from env_utils import (
    DEEPSEEK_API_KEY,
    DEEPSEEK_BASE_URL,
    DEEPSEEK_MODEL_NAME_RE,
    DEEPSEEK_TEMPERATURE  # 导入 float 类型的温度值
)

# 也可以赋值给环境变量使用
os.environ["OPENAI_API_KEY"] = DEEPSEEK_API_KEY
os.environ["OPENAI_BASE_URL"] = DEEPSEEK_BASE_URL
# 任何大语言模型都匹配 openai
llm = ChatOpenAI(
    model_name=DEEPSEEK_MODEL_NAME_RE,
    temperature=DEEPSEEK_TEMPERATURE,
)

response = llm.invoke('今天天气怎么样？')

output_str = StrOutputParser()

response_str = output_str.invoke(response)

print(response_str)
print(type(response_str))  # <class 'str'>


--- DeepSeek 配置信息 ---
API Key (前5位): sk-75...
Base URL: https://api.deepseek.com
Model Name: deepseek-reasoner
DEEPSEEK_TEMPERATURE (float): 0.5

配置成功，可以开始调用 DeepSeek API。
要查询今天的具体天气情况，我需要知道您所在的**城市名称**哦！😊

由于我无法获取实时位置信息，您可以：
- **直接告诉我城市名**，我会为您查询最新的天气情况
- **自己快速查询**：在天气App、搜索引擎中输入“XX天气”即可查看

例如您可以这样问我：“今天北京天气怎么样？”或者“上海现在天气如何？”

告诉我您的城市，我马上为您提供详细的天气信息！☀️🌧️❄️
<class 'str'>


# 2.JsonOutputParser

In [15]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import JsonOutputParser

output_json = JsonOutputParser()

print("output_json.get_format_instructions():", output_json.get_format_instructions())
chat_prompt = ChatPromptTemplate(
    messages=[
        ("system", '你是一个{name}'),
        ("human", '我有一个{question},{type}')
    ],
    partial_variables={
        'type': output_json.get_format_instructions()
    }
)

format_message = chat_prompt.invoke(
    {"name": '人工智能', "question": "人工智能用英语怎么说的问题。"})

print("format_message", format_message)
response = llm.invoke(format_message)

response_json = output_json.invoke(response)

print(response_json)
print(type(response_json))  # <class 'dict'>

output_json.get_format_instructions(): Return a JSON object.
format_message messages=[SystemMessage(content='你是一个人工智能', additional_kwargs={}, response_metadata={}), HumanMessage(content='我有一个人工智能用英语怎么说的问题。,Return a JSON object.', additional_kwargs={}, response_metadata={})]
{'original_question': '我有一个人工智能用英语怎么说的问题。', 'translated_question': "How to say 'artificial intelligence' in English?", 'answer': 'artificial intelligence'}
<class 'dict'>


# 3.XMLOutputParser  (返回的也是JSON格式)

In [17]:
from langchain_core.output_parsers import XMLOutputParser

output_xml = XMLOutputParser()

chat_prompt = ChatPromptTemplate(
    messages=[
        ("system", '你是一个{name}'),
        ("human", '我有一个{question},{type}')
    ],
    partial_variables={
        'type': output_xml.get_format_instructions()
    }
)

# format_message = chat_prompt.invoke(
#     {"name": '人工智能', "question": "人工智能用英语怎么说的问题。"})


chain = chat_prompt | llm | output_xml

response = chain.invoke({"name": '人工智能', "question": "人工智能用英语怎么说的问题。"})
print(response)

{'response': [{'question': '我有一个人工智能用英语怎么说的问题。'}, {'answer': 'In English, "人工智能" is said as "artificial intelligence".'}]}


# 4.列表解析器 CommaSeparatedListOutputParser -> 通常生成cvs

In [19]:
from langchain_core.output_parsers import CommaSeparatedListOutputParser

# 1. 初始化解析器
parser = CommaSeparatedListOutputParser()

# 2. 获取格式指令
# 这些指令告诉 LLM 如何格式化输出 (例如：使用逗号分隔)
format_instructions = parser.get_format_instructions()
print("--- 列表格式指令 ---")
print(format_instructions)

# 3. 创建 Prompt
SYSTEM_TEMPLATE = """
你是一个列表生成器。
请严格遵循以下指令输出：

{format_instructions}
"""

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", SYSTEM_TEMPLATE),
        ("human", "请为我生成 5 个与 {topic} 相关的高频关键词。")
    ]
)

# 5. 准备输入数据
input_data = {
    "topic": "人工智能在医疗领域的应用",
    # 关键：将格式指令作为变量传入 Prompt
    "format_instructions": format_instructions
}

chain = prompt | llm | parser

response = chain.invoke(input_data)

print(response)
print(type(response))  # <class 'list'>


--- 列表格式指令 ---
Your response should be a list of comma separated values, eg: `foo, bar, baz` or `foo,bar,baz`
['Medical Imaging Analysis', 'Disease Diagnosis', 'Drug Discovery', 'Personalized Medicine', 'Health Monitoring']
<class 'list'>


# 5.时间解析器 DatetimeOutputParser

In [21]:
from langchain_classic.output_parsers import DatetimeOutputParser

# 1. 初始化解析器
# 指定格式为 YYYY-MM-DD HH:MM:SS (例如: 2025-12-25 10:30:00)
DATETIME_FORMAT = "%Y-%m-%d %H:%M:%S"
parser = DatetimeOutputParser(format=DATETIME_FORMAT)

# 2. 获取格式指令
# 这些指令将明确告诉 LLM 必须严格遵循 DATETIME_FORMAT 输出时间。
format_instructions = parser.get_format_instructions()

print("--- 生成的时间格式指令 ---")
print(format_instructions)

# 3. 创建 Prompt
SYSTEM_TEMPLATE = """
你是一个时间信息提取专家。
你的任务是从用户提供的文本中准确提取时间信息，并严格按照以下指令生成时间字符串。

{format_instructions}
"""

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", SYSTEM_TEMPLATE),
        ("human", "请告诉我，文件上传成功的时间是什么时候？文本：{text}")
    ]
)
# 5. 准备输入数据
text_containing_time = "用户报告在今天下午 3 点 45 分，具体来说是 2025 年 12 月 26 日，文件上传已完成。"

input_data = {
    "text": text_containing_time,
    # 关键：将格式指令作为变量传入 Prompt
    "format_instructions": format_instructions
}

chain = prompt | llm | parser

response = chain.invoke(input_data)
print(response)
print(type(response))  # <class 'datetime.datetime'>

--- 生成的时间格式指令 ---
Write a datetime string that matches the following pattern: '%Y-%m-%d %H:%M:%S'.

Examples: 2025-12-26 09:47:04, 2024-12-26 09:47:04, 2025-12-25 09:47:04

Return ONLY this string, no other words!
2025-12-26 15:45:00
<class 'datetime.datetime'>
